# Foreword

This notebook is a complete example of a mapping pipeline which build ETL scripts to transform/map the source data into OMOP CDM tables. \
It is a reference implementation of the mapping guideline.

## Illustration of the following mapping flow

<img src="../../images/mapping_flow.svg" alt="Mapping flow" width="4000"/>


Note that your own source data set may be very different from the presented example, It is only meant as an illustration of the mapping guideline.
The example contains many special cases which are unlikely to coexist together in a real dataset, this allow us to demonstrate many mapping situations.

More specifically, you may also need to add variable transformation steps into your pipeline. 


# Import and load

In [1]:
# import
from pathlib import Path

from afbstepomop.source_validator.structural import load_structural
from afbstepomop.source_validator.vocabulary import load_vocabulary
from afbstepomop.source_validator.spec import load_spec
from afbstepomop.source_validator.display import describe, describe_minimal

from afbstepomop.output_validator.validate import read_dataset, validate

import pandas as pd

In [2]:
# load AFBSTEP-OMOP specifications, vocabulary, and structural definitions

schema = load_structural()
spec = load_spec()
vocabulary = load_vocabulary()

In [3]:
# load the reference source data set (4 tables)

SRC, OUT = Path("initial_dataset"), Path("reference_mapped_dataset")

# keep_default_na=False matters: the source writes a literal "NA" as its
# missingness marker for Case 5, and pandas' default na_values would read
# that as a null - silently turning "missing, cause unknown" into "never
# assessed", which is a different clinical statement.
read = lambda name: pd.read_csv(SRC / name, dtype=str, keep_default_na=False)

ini_baseline = read("reference_baseline.csv")        # wide, 1 row per person
ini_followup = read("reference_followup.csv")        # long, 1 row per record
ini_devices  = read("reference_device_registry.csv") # 1 row per device
ini_study    = read("reference_study_information.csv")  # long, 1 row per fact

# Step 0: explore

## Reference data set

In [4]:
# Four tables, split by who holds the data:
#   baseline / followup   the site's EDC (REDCap-style) export
#   device_registry       the CIED vendor / device desk
#   study_information     the study coordinator's metadata portal

# Quick look the tables shape
for name, df in [("baseline", ini_baseline), ("followup", ini_followup),
                 ("devices", ini_devices), ("study", ini_study)]:
    print(f"{name}: {len(df)} rows x {len(df.columns)} columns")
print()

# the baseline table is wide, with repeated field groups
print("baseline column groups:")
for group in ("dev", "med", "hist"):
    cols = [c for c in ini_baseline.columns if c.startswith(group)]
    print(f"  {group:5s} {len(cols):>2} columns -> {cols[:3]} ...")

baseline: 26 rows x 120 columns
followup: 221 rows x 26 columns
devices: 30 rows x 13 columns
study: 120 rows x 6 columns

baseline column groups:
  dev    3 columns -> ['dev1_ref', 'dev2_ref', 'dev3_ref'] ...
  med   12 columns -> ['med1_name', 'med1_class', 'med1_start'] ...
  hist  40 columns -> ['hist1_case_no', 'hist1_kind', 'hist1_start'] ...


In [5]:
# the follow-up table is long: rec_kind says what each row is
ini_followup.rec_kind.value_counts()

rec_kind
lab          73
window       50
admission    23
device       22
episode      17
procedure    14
daily         7
med           6
file          5
death         4
Name: count, dtype: int64

In [6]:
# study_information carries three different grains in one long file
ini_study.entry_kind.value_counts()

entry_kind
meta          72
enrollment    23
arm           10
country        8
provision      7
Name: count, dtype: int64

## AFBSTEP-OMOP model

In [7]:
# Let's now look at the AFBSTEP-OMOP model

# This shows the list of target tables
print(describe(spec))
print("\n\n")

AFBSTEP project layer
17 tables in scope, 53 field requirements, 46 value ranges. A table not listed is out of scope.

table                 tier       fields  ranges  rationale
────────────────────  ─────────  ──────  ──────  ────────────────────────────────────────────────────
device_specs          expected   6       0       AFBSTEP extension recording the device and detectio…
episode_event         expected   3       0       Links an episode to the measurement and observation…
visit_occurrence      expected   1       0       Baseline visit ascertainment (concept 2000013017 ==…
condition_occurrence  mandatory  8       0       Carries the baseline comorbidities and the study en…
episode               mandatory  8       0       Container for AF-burden assessment. One row per agg…
measurement           mandatory  6       41      Carries the numeric AF burden (percent time in AF) …
observation           mandatory  5       5       Carries how the burden was assessed: estimation met…
observ

In [8]:
# The entire list of minimal data set for AFBSTEP-OMOP, with all columns and their bindings
print(describe_minimal(spec, vocabulary))


AFBSTEP minimal data set
79 items over 72 distinct concepts that a conforming export must contain, and the field each belongs in.

AB — AF burden  (11 items)
item                                            concept_id  belongs in
──────────────────────────────────────────────  ──────────  ──────────────────────────────────
Analysable ECGs in window                       2000008008  observation.observation_concept_id
Atrial fibrillation burden at baseline          2000008002  measurement.measurement_concept_id
Atrial fibrillation burden during follow-up     2000008002  measurement.measurement_concept_id
Baseline burden assessment modality             2000009000  observation.observation_concept_id
Baseline monitoring period                      2000011002  episode.episode_concept_id
Burden estimation method                        2000008003  observation.observation_concept_id
ECGs classified as atrial fibrillation in win…  2000008009  observation.observation_concept_id
Episode duration   

In [9]:
# Filter the minimal data set to a single domain or table, for a more focused view
print("List of minimal concepts for the DS/DD domain (one questionnaire domain):\n")
print(describe_minimal(spec, vocabulary, "DS/DD"))        # one questionnaire domain
print("\n\n")

print("List of minimal concepts for the episode table: \n")
print(describe_minimal(spec, vocabulary, "episode"))      # everything that lands in one table


List of minimal concepts for the DS/DD domain (one questionnaire domain):

AFBSTEP minimal data set
3 items over 3 distinct concepts that a conforming export must contain, and the field each belongs in.

DS/DD — Disposition, death  (3 items)
item              concept_id  belongs in
────────────────  ──────────  ──────────────────────────────────
Death             44803317    death.death_type_concept_id
End of follow-up  44810922    observation.observation_concept_id
Last contact      3016125     observation.observation_concept_id



List of minimal concepts for the episode table: 

AFBSTEP minimal data set
2 items over 1 distinct concepts that a conforming export must contain, and the field each belongs in.

AB — AF burden  (2 items)
item                         concept_id  belongs in
───────────────────────────  ──────────  ──────────────────────────
Baseline monitoring period   2000011002  episode.episode_concept_id
Follow-up monitoring period  2000011002  episode.episode_concept_id


# Step 1: frame and scope

In [10]:
# Look at the list of tables in scope. This are the target tables where the source data will be mapped to.
print("in scope:", spec.in_scope())
print("\n\n")

# Look at the columns for one table in scope 
print(describe(spec, "condition_occurrence"))

in scope: ('condition_occurrence', 'death', 'device_exposure', 'device_specs', 'drug_exposure', 'episode', 'episode_event', 'measurement', 'observation', 'observation_period', 'person', 'person_study', 'procedure_occurrence', 'source', 'study', 'study_attribute', 'visit_occurrence')



condition_occurrence
tier: mandatory

column                       requirement  rationale
───────────────────────────  ───────────  ────────────────────────────────────────────────────────────
condition_occurrence_id      required     Primary key.
person_id                    required     Foreign key to person.
condition_concept_id         required     The finding itself. Bound to the pinned condition concept s…
condition_start_date         required     Places the condition relative to the index date; every endp…
condition_end_date           expected     Distinguishes a resolved condition from an ongoing one. Rou…
condition_type_concept_id    required     Provenance of the record. Pinned to 'EHR encounte

In [11]:
# Decide per source variable if it should be mapped, is out of scope, or
# was not collected. All three are valid answers.

scope = {
    "baseline demographics":   "person, observation_period",
    "baseline comorbidities":  "condition_occurrence (+ observation for a negative)",
    "baseline labs":           "measurement (+ observation for mEHRA)",
    "baseline med slots":      "drug_exposure",
    "baseline hist slots":     "visit_occurrence, procedure_occurrence, condition_occurrence",
    "followup lab":            "measurement / observation",
    "followup med":            "drug_exposure",
    "followup admission":      "visit_occurrence + condition_occurrence",
    "followup procedure":      "procedure_occurrence",
    "followup device":         "device_exposure",
    "followup window/episode": "episode + measurement + observation + device_specs",
    "followup daily":          "OUT OF SCOPE - per-day burden has no pinned concept",
    "followup file":           "source",
    "followup death":          "death",
    "device_registry":         "device_exposure + device_specs",
    "study_information":       "study, study_attribute, person_study",
    "af_type / smoking":       "observation (question + answer concept)",
    "cha2ds2vasc, height, weight": "measurement",
}

# Step 2: route source variables to CDM tables

In [12]:
# The routing table is a simple reference for the source variable to target table/field mapping. 
# It is not used in the code, but it is useful for documentation and planning.
# Example: 
# - baseline "sex" is mapped to the "person" table, field "gender_concept_id"
# - device_registry "dev_kind" is mapped to the "device_exposure" table, field "device_concept_id" 

routing = pd.DataFrame([
    ("baseline", "sex",                "person",               "gender_concept_id"),
    ("baseline", "yob",                "person",               "year_of_birth"),
    ("baseline", "race / ethnicity",   "person",               "race_concept_id"),
    ("baseline", "incl_date",          "observation_period",   "observation_period_start_date"),
    ("baseline", "last_contact_date",  "observation_period",   "observation_period_end_date"),
    ("baseline", "<c>_yn = 1",         "condition_occurrence", "condition_concept_id"),
    ("baseline", "<c>_yn = 0",         "observation",          "value_as_concept_id"),
    ("baseline", "<c>_datecert",       "condition_occurrence", "condition_start_date"),
    ("baseline", "krea / hb / lvef",   "measurement",          "value_as_number"),
    ("baseline", "*_flag",             "measurement/observation", "value_as_concept_id"),
    ("baseline", "ehra_class",         "observation",          "value_as_concept_id"),
    ("baseline", "smoking",            "observation",          "value_as_concept_id"),
    ("baseline", "height / weight / cha2ds2vasc", "measurement", "value_as_number"),
    ("baseline", "medN_*",             "drug_exposure",        "drug_concept_id"),
    ("baseline", "histN_* admission",  "visit_occurrence",     "visit_concept_id"),
    ("baseline", "histN_* procedure",  "procedure_occurrence", "procedure_concept_id"),
    ("baseline", "histN_* complication", "condition_occurrence", "condition_concept_id"),
    ("followup", "window",             "episode",              "episode_concept_id"),
    ("followup", "episode",            "episode",              "episode_concept_id"),
    ("followup", "af_burden",          "measurement",          "value_as_number"),
    ("followup", "hr_max",             "measurement",          "value_as_number"),
    ("followup", "rhythm",             "observation",          "value_as_concept_id"),
    ("followup", "file",               "source",               "file_concept_id"),
    ("followup", "death",              "death",                "cause_concept_id"),
    ("devices",  "dev_kind",           "device_exposure",      "device_concept_id"),
    ("devices",  "serial_no / algo",   "device_specs",         "device_sn"),
    ("study",    "meta",               "study",                "study_type_concept_id"),
    ("study",    "country / provision","study_attribute",      "value_concept_id"),
    ("study",    "enrollment",         "person_study",         "arm_type_concept_id"),
], columns=["source_table", "source_variable", "cdm_table", "cdm_field (column)"])

routing

,source_table,source_variable,cdm_table,cdm_field (column)
0,baseline,sex,person,gender_concept_id
1,baseline,yob,person,year_of_birth
2,baseline,race / ethnicity,person,race_concept_id
3,baseline,incl_date,observation_period,observation_period_start_date
4,baseline,last_contact_date,observation_period,observation_period_end_date
5,baseline,<c>_yn = 1,condition_occurrence,condition_concept_id
6,baseline,<c>_yn = 0,observation,value_as_concept_id
7,baseline,<c>_datecert,condition_occurrence,condition_start_date
8,baseline,krea / hb / lvef,measurement,value_as_number
9,baseline,*_flag,measurement/observation,value_as_concept_id


# Step 3: map source labels to concepts

This is by far the most difficult part.
Each of the source variables must be mapped into a relevant OMOP code. \
This task is basically a loopup task. 

It can be partially automated using some 3th party tools (see Readme).

The presented example has variable that directly maps to OMOP code, you may need to perform some additional transformation setps. For example, if you are mapping into the CHADS2 score (code [4229700](https://athena.ohdsi.org/search-terms/terms?query=+CHADS2+score)), you will first need to calculate the score using the appropriate conditions ( Congestive heart failure, Hypertension, Age, Diabetes mellitus, Prior Stroke or TIA or Thromboembolism)

In [13]:
# The source uses in-house shorthand throughout, so this step is a real
# lookup rather than a rename. Every dictionary below is one mapping
# decision.


UNMAPPED = 0                       # OMOP's "no matching concept"

# --- demographics ---------------------------------------------------------
GENDER    = {"M": 8507, "F": 8532, "D": 2000014000}
RACE      = {"white": 8527, "asian - indian": 38003574}
ETHNICITY = {"italian": 1546579, "east indian": 1546388}
SMOKING   = {"former": 45883458, "current": 36309332, "never": 45879404}
SMOKING_QUESTION = 43054909

# --- provenance / type concepts -------------------------------------------
EHR_ENCOUNTER, CRF, EHR_EPISODE = 32827, 32809, 32828
DEVICE_TYPE, WEARABLE_TYPE      = 32817, 705183
ENROLMENT_PERIOD                = 44814723
CONFIRMED_DIAGNOSIS             = 32893
DRUG_TYPE                       = 32809
BASELINE_VISIT                  = 2000013017   # AFBSTEP: marks a visit_occurrence row as baseline
VISIT_ID_FIELD                  = 1147869      # field concept for visit_occurrence.visit_occurrence_id

# --- comorbidities --------------------------------------------------------
COMORBIDITY = {"htn": 316866, "dm": 201820, "hf": 316139,
               "stroke": 373503,        # transient cerebral ischemia
               "ckd": 46271022}
ICD10 = {"I48.0": 313217, "I48.1": 313217, "I50.9": 316139,
         "I63.9": 443454,               # cerebral infarction
         "R55": UNMAPPED, "D62": UNMAPPED,
         "COMPL_PERIPROC": 2000005000}  # peri-procedural complication

# Case 3 of the comorbidity-dating rules: a historical diagnosis whose true
# onset date the source does not know gets this sentinel, never an empty date.
DATE_UNKNOWN_SENTINEL = "1900-01-01"
DATE_ONGOING_SENTINEL = "2099-12-31"

# --- missingness markers --------------------------------------------------
MISSING = {"not done": 45884199,          # Case 3, test not performed
           "uninterpretable": 45880382,   # Case 4, unable to determine
           "NA": 2000013000}              # Case 5, missing cause unknown
FINDING_ABSENT = 4189457                  # Case 2, explicitly negative
# Case 1 (never assessed) is the absence of a row and needs no concept.

VITALS = {"sbp_mmhg": (3004249, 8876),
          "dbp_mmhg": (3012888, 8876),
          "hr_bpm":   (3027018, 8483)}
EXTRA_LABS = {"egfr_ml_min":    (40764999, None),
              "ntprobnp_pg_ml": (3029187, 8845),
              "ldl_mmol_l":     (3001308, 8753)}
FU_VITALS = {"sbp": (3004249, 8876), "dbp": (3012888, 8876),
             "hr": (3027018, 8483), "egfr": (40764999, None),
             "ntprobnp": (3029187, 8845), "ldl": (3001308, 8753)}
CHA2DS2_VA = 2000002000
ECG_RHYTHM = 3022318
ECG_ANSWER = {"AF": 313217, "AFL": 314665, "SR": 45877096, "paced": 45877359}
AF_TIME_SINCE_DX     = 2000008000
AF_DEVICE_DETECTED   = 2000000001
MONITORING_ADHERENCE = 2000007003
ASCERTAINMENT_SOURCE = 2000007002

# --- measurements ---------------------------------------------------------
MEASURES = {"krea": (3016723, 8840),      # serum creatinine, mg/dL
            "hb":   (3000963, 8636),      # haemoglobin, g/L
            "lvef": (3027172, 8554)}      # LVEF, percent
BASELINE_MEASURES = {"krea_mgdl": "krea", "hb_gl": "hb", "lvef_pct": "lvef"}
BODY = {"height_cm": (3036277, 8582), "weight_kg": (3025315, 9529),
        "cha2ds2vasc": (37017409, None)}

# --- mEHRA symptom class --------------------------------------------------
AF_PATTERN_QUESTION = 2000000002
AF_PATTERN = {"paroxysmal": 4154290, "persistent": 4232697,
              "long-standing persistent": 45768480, "permanent": 4232691}

EHRA_QUESTION = 2000009007
EHRA = {"I": 2000009008, "IIa": 2000009009, "IIb": 2000009010,
        "III": 2000009011, "IV": 2000009012}

# --- procedures (OPS -> concept) ------------------------------------------
AF_ABLATION, CIED_IMPLANT = 2000003000, 2000003006
OPS = {"8-835.90": AF_ABLATION, "8-835.91": 45769227,
       "8-835.92": AF_ABLATION, "8-835.93": AF_ABLATION,
       "8-835.99": AF_ABLATION,
       "8-835.94": 2000003001, "8-835.95": 4323629,
       "8-640.0": 4078793, "5-35a.41": 37208174,
       "5-377.5": CIED_IMPLANT, "5-377.6": CIED_IMPLANT,
       "5-377.82": CIED_IMPLANT, "5-377.83": CIED_IMPLANT,
       "5-377.0": UNMAPPED}             # GAP: ILR implantation not pinned

# --- devices --------------------------------------------------------------
DEVICE = {"PM": 4030875, "CRT-P": 45767329, "CRT-D": 45767328,
          "S-ICD": 37164898, "ILR": 1448963, "HOLTER": 45762052,
          "PATCH": 45877787, "PPG-WATCH": 2000004006,
          "TELE-ECG": 45762462, "REST-ECG": 45768113,
          "CIED-unspec": 2000004000, "CIED-monitor": 45877787}
ENERGY_SOURCE = {"RF": 2000004001, "Cryo": 2000004002, "PFA": 2000004007,
                 "Laser": 2000004003, "n.s.": 2000004004,
                 "PVI-other": 2000004005}
WEARABLE_KINDS = {"PPG-WATCH", "PATCH"}

# --- drugs ----------------------------------------------------------------
DRUG = {"VKA": 2000012000, "DOAC": 4186985, "betablocker": 4189269,
        "antiarrhythmic": 4187146, "diuretic": 4186999,
        "RAS-I": 3666997, "SGLT2-I": 1123627, "plt-inhibitor": 36675126,
        "digitalis": 4249287, "CCB": 4159947, "MRA": 4156739,
        "statin": 3170518, "ARNI": 1588648}

# --- AF burden / episodes -------------------------------------------------
AGGREGATED_WINDOW, DISAGGREGATED_EPISODE = 2000011002, 2000011003
AF_BURDEN, PERCENT       = 2000008002, 8554
MAX_RATE, COUNTS_PER_MIN = 2000009004, 8483
EPISODE_RHYTHM           = 2000009003
RHYTHM = {"AF": 313217, "AFL": 314665, "AT": 4171269}

# --- external files -------------------------------------------------------
FILE_KIND = {"report-pdf": 2000010002, "raw-cied": 2000010004,
             "waveform": 2000010003, "interrog-pdf": 2000010005,
             "audio": 2000010001, "scan-pdf": 2000010000}
EPISODE_FIELD, MEASUREMENT_FIELD = 798885, 1147138
OBSERVATION_FIELD, DRUG_FIELD    = 1147165, 1147707

# --- disposition / demographics required by the minimal data set ----------
DEATH_TYPE, END_OF_FOLLOWUP, LAST_CONTACT = 44803317, 44810922, 3016125
ENROLMENT_SETTING = 36304160          # Encounter location [Type]
SETTING = {"amb": 9202, "stat": 9201}

# --- how the burden was assessed ------------------------------------------
ASSESSMENT_MODALITY  = 2000009000     # question: mode of AF-burden assessment
ESTIMATION_METHOD    = 2000008003     # question: AF burden estimation method
CONTINUOUS_CIED, INTERMITTENT_ECG = 2000008004, 2000008005
N_ECG_TOTAL, N_ECG_AF = 2000008008, 2000008009

# --- death ----------------------------------------------------------------
DEATH_CAUSE = {"cardiovascular, unspec.": 2000006001, "malignancy": 443392,
               "MI": 4329847, "stroke": 2000006002}

# --- study metadata -------------------------------------------------------
STUDY_KIND = {"RCT": 2000013001, "IDE": 2000013002, "OBS-COHORT": 2000013003,
              "REGISTRY-RPM": 2000013004, "CLAIMS": 2000013005,
              "OTHER": 2000013006}
DS_STATUS  = {"locked": 2000013007, "rolling": 2000013008}
TRANSFER   = {"single": 2000013009, "periodic": 2000013010}
PROVISION  = {"META": 2000013011, "TAB": 2000013012,
              "TAB+SIG": 2000013013, "TAB+IMG": 2000013014}
ATTR_TYPE  = {"provision": 2000013015, "country": 2000013016}
COUNTRY = {"IT": 41987173, "IN": 4320154, "DE": 4330435, "NL": 4331168,
           "ES": 4330441, "DK": 4330437, "FR": 4330442, "SE": 4331170}
ARM_ROLE = {"treatment": 37529738, "control": 37542561,
            "placebo": 37530454, "sham": 37543327,
            "no intervention": 37529223}

person_id = {pid: i for i, pid in enumerate(sorted(ini_baseline.pat_id), start=1)}
print(f"{len(person_id)} persons, {len(DEVICE) + len(ENERGY_SOURCE)} device concepts, "
      f"{len(RHYTHM)} rhythm concepts")

26 persons, 18 device concepts, 3 rhythm concepts


# Step 4: resolve dates and identifiers

The `observation_period` and `visit_occurence` are the OMOP tables that track high level date information.

In addition, some variables and their related dates may live in different tables. This step aims at bringing the variables and their related dates "together"

In [14]:
# Create the OMOP observation_period table, one row per enrolled person. 
# Patients with missing enrolment date get no observation_period row.

# extract inclusion date
enrolled = ini_baseline[ini_baseline.incl_date != ""].copy()
# add mapped person_id column
enrolled["person_id"] = enrolled.pat_id.map(person_id)

# create observation_period table, one row per enrolled person.
observation_period = pd.DataFrame({
    "observation_period_id": range(1, len(enrolled) + 1),             # Unique identifier for each observation period
    "person_id": enrolled.person_id.values,                           # Foreign key to the person table
    "observation_period_start_date": enrolled.incl_date.values,       # Start date of the observation period is the inclusion date
    "observation_period_end_date": enrolled.last_contact_date.values, # End date of the observation period is the last contact date
    "period_type_concept_id": ENROLMENT_PERIOD,                       # Use the concept ID for "enrollment period" to indicate the type of observation period
})

# check how many enrolled persons have no observation period (i.e., missing inclusion date)
period_of = dict(zip(enrolled.pat_id, observation_period.observation_period_id))
incl_of   = dict(zip(ini_baseline.pat_id, ini_baseline.incl_date))

print(f"{len(observation_period)} observation periods; "
      f"no period for {sorted(set(ini_baseline.pat_id) - set(period_of))}")

23 observation periods; no period for ['P-004', 'P-023', 'P-024']


In [15]:
# Visits come from every admission, wherever it sits: 
# - a single-day stay is an outpatient visit
# - a multi-day stay an inpatient one - which reproduces
# the codebook's `visit_occurrence` rows 1-5 exactly.

# count how many histN_* groups there are in the baseline table
# each histN_* group is a set of columns that describe one historical admission, procedure, or complication in the wide format baseline table
N_HIST = sum(1 for c in ini_baseline.columns
             if c.startswith("hist") and c.endswith("_case_no"))

# function to convert wide to long format for historical events (admissions, procedures, complications)
def baseline_events(kind):
    """Unstack the histN_* column groups back into one row per event."""
    out = []
    for _, r in ini_baseline.iterrows():
        for i in range(1, N_HIST + 1):
            if r[f"hist{i}_case_no"] and r[f"hist{i}_kind"] == kind:
                out.append({"pat_id": r.pat_id,
                            **{k: r[f"hist{i}_{k}"] for k in
                               ("case_no", "start", "end", "code_sys", "code",
                                "code_txt", "dept", "energy_src", "dev_ref")}})
    return pd.DataFrame(out)

# Start with follow-up table
fu = ini_followup

# create the admissions table by combining historical admissions from the baseline table and follow-up admissions from the follow-up table. 
# The resulting DataFrame is sorted by patient ID and start date, and reset index for clean indexing.
admissions = pd.concat([
    baseline_events("admission"),
    fu[fu.rec_kind == "admission"].rename(
        columns={"rec_id": "case_no", "ts_start": "start", "ts_end": "end"})
      [["pat_id", "case_no", "start", "end", "code_sys", "code", "code_txt", "dept"]],
], ignore_index=True).sort_values(["pat_id", "start"]).reset_index(drop=True)

# Add keys/indices 
admissions["person_id"] = admissions.pat_id.map(person_id)
admissions["visit_occurrence_id"] = range(1, len(admissions) + 1)

# Determine visit_concept_id based on whether the admission is a single-day stay (outpatient) or multi-day stay (inpatient)
# OMOP code: 9201 = Inpatient Visit, 9202 = Outpatient Visit
admissions["visit_concept_id"] = [
    9202 if s == e else 9201 for s, e in zip(admissions.start, admissions.end)]

# store the mapping of case numbers (source IDs) to visit_occurrence_id for later use
visit_of = dict(zip(admissions.case_no, admissions.visit_occurrence_id))
# store the mapping of patient IDs to their baseline visit_occurrence_id for later use
#
# The baseline visit is the person's visit closest in time to their enrolment
# date, not merely a visit that happens to fall on it. The old rule compared
# the two dates for equality, so it found a baseline visit for exactly one
# person out of 26; every other site records its visits days or months off the
# enrolment date, and those persons were silently left without one.
#
# Ties (two visits equidistant from enrolment) resolve to the earlier one:
# `admissions` is already sorted by (pat_id, start) and `min` keeps the first
# minimum it sees. There is no such tie in the reference data.
#
# A person with no visit at all, or with no enrolment date, gets no entry -
# the codebook's "when to omit it" case, where comorbidity Case 1 falls back
# to the enrolment date.
# A visit only counts as a baseline-visit candidate if it is BOTH:
# - outpatient, and
# - within BASELINE_WINDOW_DAYS of the enrolment date.
# Nearest-by-date alone was not enough: it picked whichever admission happened
# to be closest, and for most persons that was a ward stay months or years
# later - a follow-up event mislabelled as baseline. The type test is the
# stronger of the two; the window keeps a genuine outpatient visit from being
# claimed as baseline when it is really a later check-up.
BASELINE_WINDOW_DAYS = 30

baseline_visit, baseline_visit_date = {}, {}
for pat, visits in admissions.groupby("pat_id"):
    incl = incl_of.get(pat)
    if not incl:
        continue
    candidates = [v for v in visits.itertuples()
                  if v.start == v.end                      # single-day, i.e. 9202 outpatient
                  and abs((pd.Timestamp(v.start) - pd.Timestamp(incl)).days)
                      <= BASELINE_WINDOW_DAYS]
    if not candidates:
        continue
    nearest = min(candidates,
                  key=lambda v: abs(pd.Timestamp(v.start) - pd.Timestamp(incl)))
    baseline_visit[pat] = nearest.visit_occurrence_id
    baseline_visit_date[pat] = nearest.start

# create the visit_occurrence table in OMOP format, using the admissions DataFrame.
visit_occurrence = pd.DataFrame({
    "visit_occurrence_id": admissions.visit_occurrence_id,  # Unique identifier for each visit occurrence
    "person_id": admissions.person_id,                      # Foreign key to the person table
    "visit_concept_id": admissions.visit_concept_id,        # Concept ID indicating the type of visit (inpatient or outpatient)
    "visit_start_date": admissions.start,                   # Start date of the visit occurrence
    "visit_end_date": admissions.end,                       # End date of the visit occurrence
    "visit_type_concept_id": EHR_ENCOUNTER,                 # Concept ID indicating the type of visit. Here all visit are defined as EHR encounters (electronic health record)
    "visit_source_value": admissions.dept,                  # Source value for the visit occurrence. here using the department as the source value
})

visit_occurrence.head()

,visit_occurrence_id,person_id,visit_concept_id,visit_start_date,visit_end_date,visit_type_concept_id,visit_source_value
0,1,1,9201,2023-11-02,2023-11-03,32827,EPU
1,2,1,9202,2024-02-01,2024-02-01,32827,kardio-amb
2,3,2,9201,2024-02-20,2024-02-21,32827,kardio-stat
3,4,3,9201,2024-03-01,2024-03-02,32827,EPU
4,5,3,9202,2024-03-15,2024-03-15,32827,kardio-amb


# Step 5: transform and build the tables

This step involves actually building the ETL processes that will map your source data into an AFBSTEP-OMOP conform final data set.

Each cell below corresponds to a target table.

In [16]:
# Person table

person = pd.DataFrame({
    "person_id":            ini_baseline.pat_id.map(person_id),                                     # Unique identifier for each person
    "gender_concept_id":    ini_baseline.sex.map(GENDER).fillna(UNMAPPED).astype(int),              # Map gender to concept ID, fill unmapped values with UNMAPPED
    "year_of_birth":        pd.to_numeric(ini_baseline.yob, errors="coerce").astype("Int64"),       # Convert year of birth to numeric, coercing errors to NaN, and then convert to Int64 type
    "race_concept_id":      ini_baseline.race.map(RACE).fillna(UNMAPPED).astype(int),               # Map race to concept ID, fill unmapped values with UNMAPPED
    "ethnicity_concept_id": ini_baseline.ethnicity.map(ETHNICITY).fillna(UNMAPPED).astype(int),     # Map ethnicity to concept ID, fill unmapped values with UNMAPPED
    "person_source_value":  ini_baseline.pat_id,                                                    # Source value for the person, using the patient ID from the baseline data
    "gender_source_value":  ini_baseline.sex,                                                       # Source value for gender, using the sex from the baseline data
    "race_source_value":    ini_baseline.race,                                                      # Source value for race, using the race from the baseline data
    "ethnicity_source_value": ini_baseline.ethnicity,                                               # Source value for ethnicity, using the ethnicity from the baseline data
})
person.head()

,person_id,gender_concept_id,year_of_birth,race_concept_id,ethnicity_concept_id,person_source_value,gender_source_value,race_source_value,ethnicity_source_value
0,1,8532,1958,0,0,P-001,F,not collected,not collected
1,2,8507,1965,8527,1546579,P-002,M,white,italian
2,3,8532,1971,38003574,1546388,P-003,F,asian - indian,east indian
3,4,8507,1990,0,0,P-004,M,not collected,not collected
4,5,8507,1941,8527,0,P-005,M,white,german


In [ ]:
# condition_occurrence table
#  
# A comorbidity flag becomes: 
# - a condition row when positive
# - an observation row carrying "clinical finding absent" when explicitly negative
# - nothing at all when the site never asked (missingness Case 1)

# creating condition_occurrence and observation records as appropriate.
conditions = []  # holds positive comorbidity observations
negatives = []   # holds explicitly negative comorbidity observations 

# Loop over the baseline table to extract comorbidities and other diagnoses
for _, r in ini_baseline.iterrows():

    pid = person_id[r.pat_id] # OMOP person_id for the current patient  
    pat = r.pat_id            # source patient ID for the current patient

    # loop over flaged comorbidities defined in the COMORBIDITY dictionary defined in step 3
    for flag, concept in COMORBIDITY.items():
        # extract the yes/no, onset date, and certification status for the comorbidity from the baseline record
        yn, onset, cert = r[f"{flag}_yn"], r[f"{flag}_onset"], r[f"{flag}_datecert"]
        # if the comorbidity is present (yn == "1"), create a condition_occurrence record with the appropriate details
        if yn == "1":
            # the three comorbidity-dating cases
            date = {"baseline": baseline_visit_date.get(pat, r.incl_date),
                    "exact": onset,
                    "unknown": DATE_UNKNOWN_SENTINEL}[cert]
            conditions.append({
                "person_id": pid, "condition_concept_id": concept,
                "condition_start_date": date,
                "condition_type_concept_id": CRF,
                "visit_occurrence_id":
                    baseline_visit.get(pat) if cert == "baseline" else None,
                "condition_source_value": f"{flag}={yn}/{cert}"})
        # if the comorbidity is explicitly absent (yn == "0"), create an observation record indicating the finding is absent
        elif yn == "0":
            negatives.append({
                "person_id": pid, "observation_concept_id": concept,
                "observation_date": r.incl_date,
                "observation_type_concept_id": CRF,
                "value_as_concept_id": FINDING_ABSENT,
                "observation_source_value": f"{flag}=0"})

    # some patients have other diagnoses coded in ICD-10 format, separated by semicolons. 
    # For each of these codes, create a condition_occurrence record with the appropriate details.
    for icd in filter(None, r.other_dx_icd10.split(";")):
        conditions.append({
            "person_id": pid, "condition_concept_id": UNMAPPED,
            "condition_start_date": r.incl_date,
            "condition_type_concept_id": CRF,
            "condition_source_value": f"ICD10:{icd}"})

# Use the admissions table to create condition_occurrence records for any diagnoses recorded during follow-up visits.
for _, e in admissions[admissions.code != ""].iterrows():
    conditions.append({
        "person_id": e.person_id,                                     # OMOP person_id for the current patient
        "condition_concept_id": ICD10.get(e.code, UNMAPPED),          # Map the ICD-10 code to a concept ID (defined in step 3), or use UNMAPPED if not found  
        "condition_start_date": e.start, "condition_end_date": e.end, # Start and end dates of the condition occurrence, taken from the admission record
        "condition_type_concept_id": EHR_ENCOUNTER,                   # Concept ID indicating the type of condition occurrence, here using EHR encounter
        "visit_occurrence_id": e.visit_occurrence_id,                 # Foreign key to the visit_occurrence table, linking the condition occurrence to the corresponding visit
        "condition_source_value": f"{e.code_sys}:{e.code}"})          # Source value for the condition occurrence, using the code system and code from the admission record

# The baseline table also contains historical complications, which are treated as condition occurrences.
# Same as above, we map the ICD-10 codes to concept IDs and create condition_occurrence records for each complication.
for _, e in baseline_events("complication").iterrows():
    conditions.append({
        "person_id": person_id[e.pat_id],                            
        "condition_concept_id": ICD10.get(e.code, UNMAPPED),          
        "condition_start_date": e.start,
        "condition_type_concept_id": EHR_ENCOUNTER,
        "visit_occurrence_id": visit_of.get(e.case_no),
        "condition_source_value": f"{e.code_sys}:{e.code}"})

# Create a DataFrame for the condition_occurrence table and add an ID column
condition_occurrence = pd.DataFrame(conditions)
condition_occurrence.insert(0, "condition_occurrence_id",
                            range(1, len(condition_occurrence) + 1))

condition_occurrence.head()


,condition_occurrence_id,person_id,condition_concept_id,condition_start_date,condition_type_concept_id,condition_status_concept_id,visit_occurrence_id,condition_source_value,condition_end_date
0,1,1,201820,2024-02-01,32809,32893.0,2.0,dm=1/baseline,NaN
1,2,1,316139,2024-02-01,32809,32893.0,2.0,hf=1/baseline,NaN
2,3,2,316139,2019-11-03,32809,NaN,NaN,hf=1/exact,NaN
3,4,3,316139,1900-01-01,32809,NaN,NaN,hf=1/unknown,NaN
4,5,5,46271022,2019-07-18,32809,32893.0,6.0,ckd=1/baseline,NaN


In [18]:
# procedure_occurrence table

# The procedure_occurrence table is created by combining historical procedures from the baseline table and follow-up procedures from the follow-up table. 
# The resulting DataFrame is sorted by patient ID and start date, and reset index for clean indexing.

fu_procedure = fu[fu.rec_kind == "procedure"]\
    .rename(
        columns={"rec_id": "case_no", "ts_start": "start", "ts_end": "end",
                 "dev_ref": "dev_ref"})\
    [["pat_id", "case_no", "start", "end", "code_sys", "code", "code_txt",
    "dept", "energy_src", "dev_ref"]]

# create intermediate procedures DataFrame.
procedures = pd.concat([
    baseline_events("procedure"),              # long format DataFrame of historical procedures from the baseline table
    fu_procedure,                              # long format DataFrame of follow-up procedures from the follow-up table 
], ignore_index=True)\
    .sort_values(["pat_id", "start"])\
    .reset_index(drop=True)

# create final procedure_occurrence DataFrame in OMOP format, mapping the source data to the appropriate OMOP fields and concepts.
procedure_occurrence = pd.DataFrame({
    "procedure_occurrence_id": range(1, len(procedures) + 1),                       # Unique identifier for each procedure occurrence
    "person_id": procedures.pat_id.map(person_id),                                  # Foreign key to the person table
    "procedure_concept_id": procedures.code.map(OPS).fillna(UNMAPPED).astype(int),  # Map the procedure code to a concept ID (defined in step 3), or use UNMAPPED if not found
    "procedure_date": procedures.start,                                             # Start date of the procedure occurrence
    "procedure_type_concept_id": EHR_ENCOUNTER,                                     # Concept ID indicating the type of procedure occurrence, here using EHR encounter
    "visit_occurrence_id": procedures.case_no.map(visit_of).astype("Int64"),        # Foreign key to the visit_occurrence table, linking the procedure occurrence to the corresponding visit
    "procedure_source_value": procedures.code_sys + ":" + procedures.code,          # Source value for the procedure occurrence, using the code system and code from the procedures DataFrame
})

procedure_occurrence.head()

,procedure_occurrence_id,person_id,procedure_concept_id,procedure_date,procedure_type_concept_id,visit_occurrence_id,procedure_source_value
0,1,1,2000003000,2023-11-02,32827,1,OPS:8-835.90
1,2,2,2000003006,2024-02-20,32827,3,OPS:5-377.5
2,3,3,2000003000,2024-03-01,32827,4,OPS:8-835.90
3,4,7,2000003000,2019-06-13,32827,<NA>,OPS:8-835.90
4,5,8,2000003000,2021-03-10,32827,<NA>,OPS:8-835.99


In [19]:
# device_exposure table

# the source data contains contains some devices related information, such as:
# - Pacemaker (PM)
# - Cardiac Resynchronization Therapy (CRT-P, CRT-D)
# - Subcutaneous Implantable Cardioverter Defibrillator (S-ICD)
# - etc
# These should be mapped to the OMOP device_exposure table, which captures information about medical devices used in patient care.

# !!! check with clinician !!!
# Note that the energy source case is ambiguous, it may not be defined as a device concept, but rather a property of the procedure.

# extract the energy source for each device from the procedures DataFrame, creating a mapping of device reference to energy source.
energy_of = dict(zip(procedures.dev_ref, procedures.energy_src))

def device_concept(row):
    if row.dev_kind == "ABL-GEN":
        return ENERGY_SOURCE.get(energy_of.get(row.dev_id, ""), UNMAPPED)
    return DEVICE.get(row.dev_kind, UNMAPPED)

# Create keys/indices for the device_exposure table.
ini_devices = ini_devices.sort_values("dev_id").reset_index(drop=True)
ini_devices["device_exposure_id"] = range(1, len(ini_devices) + 1)

# store the mapping of device IDs to device_exposure_id for later use
exposure_of = dict(zip(ini_devices.dev_id, ini_devices.device_exposure_id))

device_exposure = pd.DataFrame({
    "device_exposure_id": ini_devices.device_exposure_id,                       # Unique identifier for each device exposure
    "person_id": ini_devices.pat_id.map(person_id),                             # Foreign key to the person table
    "device_concept_id": [device_concept(r) for r in ini_devices.itertuples()], # Map the device kind to a concept ID (defined in step 3), or use UNMAPPED if not found
    "device_exposure_start_date": ini_devices.in_use_from,                      # Start date of the device exposure
    "device_exposure_end_date": ini_devices.in_use_to,                          # End date of the device exposure
    "device_type_concept_id": [                                                 # Determine the device type concept ID based on whether the device kind is in WEARABLE_KINDS or not.
        WEARABLE_TYPE if k in WEARABLE_KINDS else DEVICE_TYPE                   # If it is, use WEARABLE_TYPE; otherwise, use DEVICE_TYPE. (note: both are defined in step 3)
        for k in ini_devices.dev_kind],
    "device_source_value": ini_devices.dev_kind,
})
device_exposure.head()

,device_exposure_id,person_id,device_concept_id,device_exposure_start_date,device_exposure_end_date,device_type_concept_id,device_source_value
0,1,1,4030875,2019-06-02,,32817,PM
1,2,1,2000004006,2024-01-15,,705183,PPG-WATCH
2,3,1,2000004007,2023-11-02,,32817,ABL-GEN
3,4,2,2000004000,2024-02-20,,32817,CIED-unspec
4,5,3,2000004001,2024-03-01,,32817,ABL-GEN


In [20]:
# episode and episode_event tables

# Two structurally different things share the episode table, told apart by
# episode_concept_id:
# - an aggregated monitoring window: a continuous stretch of monitoring, over
#   which AF burden is summarised (AGGREGATED_WINDOW)
# - a single disaggregated arrhythmia episode: one individual AF event
#   (DISAGGREGATED_EPISODE)
# Both arrive in the follow-up table and are told apart there by rec_kind.

EPISODE_KIND = {"window": AGGREGATED_WINDOW, "episode": DISAGGREGATED_EPISODE}

# keep only the follow-up rows that describe a window or an episode
mon = fu[fu.rec_kind.isin(EPISODE_KIND)].copy()

mon["episode_key"] = mon.rec_id

# The source writes one row per measured parameter, so a single window appears
# several times over. Deduplicate on what actually identifies a window or an
# episode: the patient, when it started and ended, and the device behind it.
windows = mon[mon.rec_kind == "window"].drop_duplicates(
    subset=["pat_id", "ts_start", "ts_end", "dev_ref"])
episodes_only = mon[mon.rec_kind == "episode"].drop_duplicates(
    subset=["pat_id", "ts_start", "ts_end", "dev_ref"])

# one row per real window or episode, then mint the episode_id key
epi = pd.concat([windows, episodes_only]).sort_values(
    ["pat_id", "ts_start"]).reset_index(drop=True)
epi["episode_id"] = range(1, len(epi) + 1)

# Every source rec_id belonging to the same window/episode maps onto that one
# episode_id. The measurement and observation cells below use this lookup to
# attach their rows to the right episode. Written as a plain nested loop for
# readability; the reference data set is small enough that its cost is moot.
episode_of = {}
for _, e in epi.iterrows():
    key = (e.pat_id, e.ts_start, e.ts_end, e.dev_ref, e.rec_kind)
    for _, m in mon.iterrows():
        if (m.pat_id, m.ts_start, m.ts_end, m.dev_ref, m.rec_kind) == key:
            episode_of[m.rec_id] = e.episode_id

episode = pd.DataFrame({
    "episode_id": epi.episode_id,                          # Unique identifier for each episode
    "person_id": epi.pat_id.map(person_id),                # Foreign key to the person table
    "episode_concept_id": epi.rec_kind.map(EPISODE_KIND),  # Which of the two kinds this row is (concepts defined in step 3)
    "episode_start_datetime": epi.ts_start,                # Start of the window or episode, full timestamp
    "episode_end_datetime": epi.ts_end,                    # End of the window or episode, full timestamp
    "episode_start_date": epi.ts_start.str[:10],           # Date part alone, which the CDM requires alongside the datetime
    "episode_end_date": epi.ts_end.str[:10],               # Date part alone, which the CDM requires alongside the datetime
    "episode_number": 1,                                   # Position within a series; this source does not number them, so 1 throughout
    "episode_object_concept_id": UNMAPPED,                 # What the episode is "about" (e.g. a disease); not recorded by this source
    "episode_type_concept_id": EHR_EPISODE,                # Provenance of the record itself (defined in step 3)
    "episode_source_value": epi.rec_id,                    # The source's own record id, kept so a mapping decision can be audited
})

# A disaggregated episode may name the monitoring window it was detected within.
# Where it does, episode_event records that parent-child link: the window is the
# episode_id, the individual episode is the event_id.
links = mon[(mon.rec_kind == "episode") & (mon.parent_rec_id != "")]
episode_event = pd.DataFrame({
    "episode_id": [episode_of[p] for p in links.parent_rec_id],  # The parent aggregated window
    "event_id": [episode_of[r] for r in links.rec_id],           # The child disaggregated episode
    "episode_event_field_concept_id": EPISODE_FIELD,             # Names the field event_id points into, i.e. episode.episode_id (defined in step 3)
}).drop_duplicates()

print(f"{len(episode)} episodes "
      f"({(epi.rec_kind == 'window').sum()} windows, "
      f"{(epi.rec_kind == 'episode').sum()} episodes), "
      f"{len(episode_event)} links")
episode.head()

37 episodes (22 windows, 15 episodes), 8 links


,episode_id,person_id,episode_concept_id,episode_start_datetime,episode_end_datetime,episode_start_date,episode_end_date,episode_number,episode_object_concept_id,episode_type_concept_id,episode_source_value
0,1,1,2000011002,2024-02-05 14:31,2024-03-11 12:06,2024-02-05,2024-03-11,1,0,32828,M-0001
1,2,1,2000011002,2024-03-11 12:06,2024-11-14 14:12,2024-03-11,2024-11-14,1,0,32828,M-0002
2,3,1,2000011003,2024-03-19 08:14,2024-03-19 08:19,2024-03-19,2024-03-19,1,0,32828,M-0003
3,4,1,2000011003,2024-03-20 14:02,2024-03-20 14:07,2024-03-20,2024-03-20,1,0,32828,M-0004
4,5,1,2000011003,2024-03-21 03:45,2024-03-21 03:48,2024-03-21,2024-03-21,1,0,32828,M-0005


In [21]:
# measurement table

# A measurement is a clinical value that was measured, mostly numeric. Four
# different parts of the source feed this one table:
# - baseline labs and body measurements, from the baseline table
# - follow-up labs, from the follow-up table
# - AF burden, one figure per aggregated monitoring window
# - peak heart rate, one figure per disaggregated episode
#
# Missingness is handled the same way throughout (see the codebook):
# - a value present            -> a plain numeric row
# - no value but a flag        -> a row with no number, carrying a missingness
#                                 marker in value_as_concept_id (Cases 3-5)
# - neither value nor flag     -> no row at all, because the site never asked
#                                 (Case 1), and an absent row is how that is said

rows = []

# baseline labs and body measurements
for _, r in ini_baseline.iterrows():
    pid = person_id[r.pat_id]

    # labs: the value, its date and its missingness flag live in three columns
    # named off the same stem, so the stem is derived from the value column
    for col, key in BASELINE_MEASURES.items():
        concept, unit = MEASURES[key]
        stem = col.rsplit("_", 1)[0] if col != "lvef_pct" else "lvef"
        value, date, flag = r[col], r[f"{stem}_date"], r[f"{stem}_flag"]
        if not value and not flag:
            continue                       # Case 1: never assessed, no row
        rows.append({
            "person_id": pid, "measurement_concept_id": concept,
            "measurement_date": date, "measurement_type_concept_id": CRF,
            "value_as_number": value or None,             # left empty when only a flag was recorded
            "unit_concept_id": unit if value else None,   # a unit without a number would assert a measurement that was never made
            "value_as_concept_id": MISSING.get(flag),     # the missingness marker, when there is one (defined in step 3)
            "measurement_source_value": col})

    for col, (concept, unit) in {**VITALS, **EXTRA_LABS}.items():
        stem = {"sbp_mmhg": "vitals", "dbp_mmhg": "vitals", "hr_bpm": "vitals",
                "egfr_ml_min": "egfr", "ntprobnp_pg_ml": "ntprobnp",
                "ldl_mmol_l": "ldl"}[col]
        value, date, flag = r[col], r[f"{stem}_date"], r[f"{stem}_flag"]
        if not value and not flag:
            continue
        rows.append({
            "person_id": pid, "measurement_concept_id": concept,
            "measurement_date": date, "measurement_type_concept_id": CRF,
            "value_as_number": value or None,
            "unit_concept_id": unit if value else None,
            "value_as_concept_id": MISSING.get(flag),
            "measurement_source_value": col})

    if r["cha2ds2va"]:
        rows.append({"person_id": pid, "measurement_concept_id": CHA2DS2_VA,
                     "measurement_date": r.incl_date,
                     "measurement_type_concept_id": CRF,
                     "value_as_number": r["cha2ds2va"],
                     "measurement_source_value": "cha2ds2va"})

    if r["ecg_rhythm"] or r["ecg_flag"]:
        rows.append({"person_id": pid, "measurement_concept_id": ECG_RHYTHM,
                     "measurement_date": r["ecg_date"],
                     "measurement_type_concept_id": CRF,
                     "value_as_concept_id": ECG_ANSWER.get(r["ecg_rhythm"])
                                            or MISSING.get(r["ecg_flag"])
                                            or UNMAPPED,
                     "measurement_source_value": f"ecg_rhythm:{r['ecg_rhythm'] or r['ecg_flag']}"})

    # body measurements: height and weight, taken at inclusion
    for col, (concept, unit) in BODY.items():
        if r[col]:
            rows.append({
                "person_id": pid, "measurement_concept_id": concept,
                "measurement_date": r.incl_date,
                "measurement_type_concept_id": CRF,
                "value_as_number": r[col], "unit_concept_id": unit,
                "measurement_source_value": col})

# follow-up numeric labs, kept only where the variable is one we map
FU_MEASURES = {**MEASURES, **FU_VITALS}
for _, m in fu[(fu.rec_kind == "lab") & (fu.variable.isin(FU_MEASURES))].iterrows():
    concept, unit = FU_MEASURES[m.variable]
    rows.append({
        "person_id": person_id[m.pat_id], "measurement_concept_id": concept,
        "measurement_date": m.ts_start, "measurement_type_concept_id": CRF,
        "value_as_number": m.value or None,
        "unit_concept_id": unit if m.value else None,
        "value_as_concept_id": MISSING.get(m.flag),
        "measurement_source_value": f"{m.event_name}:{m.variable}"})   # event and variable together, so the row can be traced back

# AF burden per aggregated window, linked back to its episode.
# measurement_event_id and meas_event_field_concept_id are OMOP's generic way of
# pointing a row at a row in another table: the first holds the key, the second
# names which field that key belongs to. Here they attach the burden figure to
# the monitoring window it summarises, using the lookup built in the cell above.
for _, m in fu[(fu.rec_kind == "window") & (fu.variable == "af_burden")].iterrows():
    rows.append({
        "person_id": person_id[m.pat_id], "measurement_concept_id": AF_BURDEN,
        "measurement_date": m.ts_end[:10],                       # a burden figure describes the whole window, so it is dated at its end
        "measurement_type_concept_id": EHR_EPISODE,
        "value_as_number": m.value or None,
        "unit_concept_id": PERCENT if m.value else None,         # AF burden is a percentage of monitored time
        "value_as_concept_id": MISSING.get(m.flag),
        "measurement_event_id": episode_of.get(m.rec_id),        # the episode_id of the window this figure came from
        "meas_event_field_concept_id": EPISODE_FIELD,            # says that measurement_event_id points at episode.episode_id
        "measurement_source_value": "window:af_burden"})

# peak rate per disaggregated episode, linked back the same way
for _, m in fu[(fu.rec_kind == "episode") & (fu.hr_max != "")].iterrows():
    rows.append({
        "person_id": person_id[m.pat_id], "measurement_concept_id": MAX_RATE,
        "measurement_date": m.ts_start[:10],                     # a peak rate describes one event, so it is dated at its start
        "measurement_type_concept_id": EHR_EPISODE,
        "value_as_number": m.hr_max, "unit_concept_id": COUNTS_PER_MIN,
        "measurement_event_id": episode_of.get(m.rec_id),        # the episode_id of the episode this rate came from
        "meas_event_field_concept_id": EPISODE_FIELD,
        "measurement_source_value": "episode:hr_max"})

# assemble the table and mint the primary key
measurement = pd.DataFrame(rows)
measurement.insert(0, "measurement_id", range(1, len(measurement) + 1))

# Lookup from "person and where the value came from" to measurement_id. The
# external-files cell further down uses it to point a stored file at the exact
# measurement it belongs to.
measurement_of_source = dict(zip(
    measurement.person_id.astype(str) + "|" + measurement.measurement_source_value,
    measurement.measurement_id))
print(f"{len(measurement)} measurements")
measurement.head()

313 measurements


,measurement_id,person_id,measurement_concept_id,measurement_date,measurement_type_concept_id,value_as_number,unit_concept_id,value_as_concept_id,measurement_source_value,measurement_event_id,meas_event_field_concept_id
0,1,1,3016723,2024-02-01,32809,1.1,8840.0,NaN,krea_mgdl,NaN,NaN
1,2,1,3004249,2024-02-01,32809,120,8876.0,NaN,sbp_mmhg,NaN,NaN
2,3,1,3012888,2024-02-01,32809,78,8876.0,NaN,dbp_mmhg,NaN,NaN
3,4,1,3027018,2024-02-01,32809,56,8483.0,NaN,hr_bpm,NaN,NaN
4,5,1,40764999,2024-02-01,32809,86,NaN,NaN,egfr_ml_min,NaN,NaN


In [22]:
# observation table

# The observation table carries facts that are not measurements: 
# - findings,
# - statuses 
# - questionnaire answers
# Most rows here are a question/answer pair, where observation_concept_id is the question ("smoking status") and
# value_as_concept_id is the answer ("former smoker"). 
# Both concepts are pinned in step 3.
#
# Seven groups of source data land here:
# - comorbidities that were explicitly ruled out (carried over from the condition_occurrence cell)
# - smoking status and EHRA symptom class, from the baseline table
# - EHRA reassessed during follow-up
# - the rhythm type of each disaggregated episode
# - disposition: enrolment setting, last contact, end of follow-up
# - how each monitoring window's burden was assessed
# - the ECG counts behind an intermittently estimated burden

obs = list(negatives)      # the explicitly-negative comorbidities from above

for _, r in ini_baseline.iterrows():
    pid = person_id[r.pat_id]

    # smoking status: the source label is the answer, mapped via SMOKING
    if r.smoking:
        obs.append({"person_id": pid, "observation_concept_id": SMOKING_QUESTION,
                    "observation_date": r.incl_date,
                    "observation_type_concept_id": CRF,
                    "value_as_concept_id": SMOKING.get(r.smoking, UNMAPPED),
                    "observation_source_value": r.smoking})

    if r.af_type:
        obs.append({"person_id": pid,
                    "observation_concept_id": AF_PATTERN_QUESTION,
                    "observation_date": r.incl_date,
                    "observation_type_concept_id": CRF,
                    "value_as_concept_id": AF_PATTERN.get(r.af_type, UNMAPPED),
                    "observation_source_value": r.af_type})

    # EHRA symptom class. A row is written when either the class or a
    # missingness flag is present: the flag is itself an answer, saying why the
    # class is absent rather than leaving the reader to guess.
    if r.ehra_class or r.ehra_flag:
        obs.append({"person_id": pid, "observation_concept_id": EHRA_QUESTION,
                    "observation_date": r.ehra_date,
                    "observation_type_concept_id": CRF,
                    "value_as_concept_id": EHRA.get(r.ehra_class)
                                           or MISSING.get(r.ehra_flag),
                    "observation_source_value": r.ehra_class or r.ehra_flag})

# EHRA is also reassessed during follow-up, and lands in the same question
for _, m in fu[(fu.rec_kind == "lab") & (fu.variable == "ehra")].iterrows():
    obs.append({"person_id": person_id[m.pat_id],
                "observation_concept_id": EHRA_QUESTION,
                "observation_date": m.ts_start,
                "observation_type_concept_id": CRF,
                "value_as_concept_id": EHRA.get(m.value) or MISSING.get(m.flag),
                "observation_source_value": f"{m.event_name}:{m.value or m.flag}"})

# the rhythm type of a disaggregated episode is an answer concept, so it
# lives in value_as_concept_id and points back at its episode
for _, m in fu[(fu.rec_kind == "episode") & (fu.rhythm != "")].iterrows():
    obs.append({"person_id": person_id[m.pat_id],
                "observation_concept_id": EPISODE_RHYTHM,
                "observation_date": m.ts_start[:10],
                "observation_type_concept_id": EHR_EPISODE,
                "value_as_concept_id": RHYTHM.get(m.rhythm, UNMAPPED),
                "observation_event_id": episode_of.get(m.rec_id),   # the episode this rhythm describes
                "obs_event_field_concept_id": EPISODE_FIELD,        # says observation_event_id points at episode.episode_id
                "observation_source_value": f"episode:{m.rhythm}"})

for _, r in enrolled.iterrows():
    if r["af_first_dx_date"]:
        days = (pd.Timestamp(r.incl_date) - pd.Timestamp(r["af_first_dx_date"])).days
        obs.append({"person_id": person_id[r.pat_id],
                    "observation_concept_id": AF_TIME_SINCE_DX,
                    "observation_date": r.incl_date,
                    "observation_type_concept_id": CRF,
                    "value_as_number": days,
                    "observation_source_value": "af_first_dx_date"})
    if r["af_device_detected"] == "1":
        obs.append({"person_id": person_id[r.pat_id],
                    "observation_concept_id": AF_PATTERN_QUESTION,
                    "observation_date": r.incl_date,
                    "observation_type_concept_id": CRF,
                    "value_as_concept_id": AF_DEVICE_DETECTED,
                    "observation_source_value": "af_device_detected"})

for _, m in fu[fu.variable == "adherence_pct"].iterrows():
    obs.append({"person_id": person_id[m.pat_id],
                "observation_concept_id": MONITORING_ADHERENCE,
                "observation_date": m.ts_end[:10],
                "observation_type_concept_id": EHR_EPISODE,
                "value_as_number": m.value, "unit_concept_id": PERCENT,
                "observation_event_id": episode_of.get(m.rec_id),
                "obs_event_field_concept_id": EPISODE_FIELD,
                "observation_source_value": "adherence_pct"})

for _, m in fu[fu.ascertain_src != ""].iterrows():
    obs.append({"person_id": person_id[m.pat_id],
                "observation_concept_id": ASCERTAINMENT_SOURCE,
                "observation_date": m.ts_start[:10],
                "observation_type_concept_id": EHR_ENCOUNTER,
                "value_as_concept_id": UNMAPPED,
                "observation_source_value": f"ascertain:{m.ascertain_src}"})

# disposition and enrolment setting, both required by the minimal data set.
# Last contact and end of follow-up carry no answer concept: the fact being
# recorded is the date itself, which observation_date already holds.
for _, r in enrolled.iterrows():
    pid = person_id[r.pat_id]
    obs.append({"person_id": pid, "observation_concept_id": ENROLMENT_SETTING,
                "observation_date": r.incl_date,
                "observation_type_concept_id": CRF,
                "value_as_concept_id": SETTING.get(r.visit_setting, UNMAPPED),
                "observation_source_value": r.visit_setting})
    obs.append({"person_id": pid, "observation_concept_id": LAST_CONTACT,
                "observation_date": r.last_contact_date,
                "observation_type_concept_id": CRF,
                "observation_source_value": "last_contact_date"})
    obs.append({"person_id": pid, "observation_concept_id": END_OF_FOLLOWUP,
                "observation_date": r.last_contact_date,
                "observation_type_concept_id": CRF,
                "observation_source_value": "end_of_followup"})

# how each window's burden was assessed: the device answers the modality
# question, and the parameter it reported answers the estimation-method one
kind_of_device = dict(zip(ini_devices.dev_id, ini_devices.dev_kind))
for _, m in fu[fu.rec_kind == "window"].drop_duplicates(
        subset=["rec_id"]).iterrows():
    if m.rec_id not in episode_of:
        continue                       # a window that produced no episode row has nothing to attach to
    pid = person_id[m.pat_id]
    obs.append({"person_id": pid, "observation_concept_id": ASSESSMENT_MODALITY,
                "observation_date": m.ts_start[:10],
                "observation_type_concept_id": EHR_EPISODE,
                "value_as_concept_id": DEVICE.get(kind_of_device.get(m.dev_ref, ""),
                                                  UNMAPPED),
                "observation_event_id": episode_of[m.rec_id],
                "obs_event_field_concept_id": EPISODE_FIELD,
                "observation_source_value": kind_of_device.get(m.dev_ref, "")})

# A window that reported an ECG count was estimated from intermittent readings;
# one that did not came from continuous device monitoring. That is what decides
# the estimation-method answer below.
intermittent = {r.rec_id for _, r in fu.iterrows() if r.variable == "n_ecg_total"}
for _, m in fu[(fu.rec_kind == "window") & (fu.variable == "af_burden")].iterrows():
    if m.rec_id not in episode_of:
        continue
    obs.append({"person_id": person_id[m.pat_id],
                "observation_concept_id": ESTIMATION_METHOD,
                "observation_date": m.ts_start[:10],
                "observation_type_concept_id": EHR_EPISODE,
                "value_as_concept_id": INTERMITTENT_ECG if m.rec_id in intermittent
                                       else CONTINUOUS_CIED,
                "observation_event_id": episode_of[m.rec_id],
                "obs_event_field_concept_id": EPISODE_FIELD,
                "observation_source_value": "window:af_burden"})

# intermittent monitoring reports counts, not a percentage: numerator and
# denominator are observations in their own right
ECG_COUNTS = {"n_ecg_total": N_ECG_TOTAL, "n_ecg_af": N_ECG_AF}
for _, m in fu[fu.variable.isin(ECG_COUNTS)].iterrows():
    obs.append({"person_id": person_id[m.pat_id],
                "observation_concept_id": ECG_COUNTS[m.variable],
                "observation_date": m.ts_start[:10],
                "observation_type_concept_id": EHR_EPISODE,
                "value_as_number": m.value,                      # a count, so the value is numeric rather than a concept
                "observation_event_id": episode_of.get(m.rec_id),
                "obs_event_field_concept_id": EPISODE_FIELD,
                "observation_source_value": m.variable})

# mark the baseline visit, where the source has one 
for pat_id, visit_id in baseline_visit.items():
    obs.append({"person_id": person_id[pat_id],
                "observation_concept_id": BASELINE_VISIT,
                "observation_date": baseline_visit_date[pat_id],  # the visit's own start date, not the enrolment date
                "observation_type_concept_id": CRF,
                "observation_event_id": visit_id,
                "obs_event_field_concept_id": VISIT_ID_FIELD,
                "observation_source_value": "baseline_visit"})

# assemble the table and mint the primary key
observation = pd.DataFrame(obs)
observation.insert(0, "observation_id", range(1, len(observation) + 1))
print(f"{len(observation)} observations")
observation.head()

350 observations


,observation_id,person_id,observation_concept_id,observation_date,observation_type_concept_id,value_as_concept_id,observation_source_value,observation_event_id,obs_event_field_concept_id,value_as_number,unit_concept_id
0,1,2,201820,2024-02-03,32809,4189457.0,dm=0,NaN,NaN,NaN,NaN
1,2,5,316866,2019-07-18,32809,4189457.0,htn=0,NaN,NaN,NaN,NaN
2,3,5,201820,2019-07-18,32809,4189457.0,dm=0,NaN,NaN,NaN,NaN
3,4,5,316139,2019-07-18,32809,4189457.0,hf=0,NaN,NaN,NaN,NaN
4,5,5,373503,2019-07-18,32809,4189457.0,stroke=0,NaN,NaN,NaN,NaN


In [23]:
# drug_exposure table

# Concomitant medication reaches the source two ways, and both are flattened
# into one intermediate frame before mapping:
# - the baseline table holds it wide, as numbered columns med1_name, med2_name
#   and so on, so the loop below has to walk the numbers
# - the follow-up table holds it long, one row per medication
#
# The source names a drug *class* rather than a specific product, so
# drug_concept_id is mapped from the class column, and the free-text product
# name is kept in drug_source_value.

# how many medN_* column groups the baseline table actually has
N_MED = sum(1 for c in ini_baseline.columns
            if c.startswith("med") and c.endswith("_name"))
drugs = []

# baseline: unpivot the numbered columns into one row per medication
for _, r in ini_baseline.iterrows():
    for i in range(1, N_MED + 1):
        if r[f"med{i}_name"]:                  # an empty name means that slot is unused
            drugs.append({"pat_id": r.pat_id, "name": r[f"med{i}_name"],
                          "cls": r[f"med{i}_class"], "start": r[f"med{i}_start"],
                          "stop": r[f"med{i}_stop"], "doc": r[f"med{i}_doc"]})

# follow-up: already one row per medication, just renamed onto the same keys
for _, m in fu[fu.rec_kind == "med"].iterrows():
    drugs.append({"pat_id": m.pat_id, "name": m.variable, "cls": m.value,
                  "start": m.ts_start, "stop": m.ts_end, "doc": m.file_path})
drugs = pd.DataFrame(drugs).sort_values(["pat_id", "start"]).reset_index(drop=True)

drug_exposure = pd.DataFrame({
    "drug_exposure_id": range(1, len(drugs) + 1),                       # Unique identifier for each drug exposure
    "person_id": drugs.pat_id.map(person_id),                           # Foreign key to the person table
    "drug_concept_id": drugs.cls.map(DRUG).fillna(UNMAPPED).astype(int),# Map the drug class to a concept ID (defined in step 3), or use UNMAPPED if not found
    "drug_exposure_start_date": drugs.start,                            # When the medication was started
    "drug_exposure_end_date": drugs.stop.replace("", DATE_ONGOING_SENTINEL),  # When it was stopped. An ongoing medication carries the 2099-12-31 sentinel, since the CDM declares this field NOT NULL
    "drug_type_concept_id": DRUG_TYPE,                                  # Provenance of the record itself (defined in step 3)
    "drug_source_value": drugs.name,                                    # The product name as the site wrote it, kept so the mapping can be audited
})

# keep the new key alongside the intermediate frame, so the external-files cell
# below can point a scanned prescription at the exact drug_exposure row
drugs["drug_exposure_id"] = drug_exposure.drug_exposure_id
drug_exposure.head()

,drug_exposure_id,person_id,drug_concept_id,drug_exposure_start_date,drug_exposure_end_date,drug_type_concept_id,drug_source_value
0,1,1,2000012000,2024-01-10,2099-12-31,32809,Marcumar
1,2,5,4186985,2019-07-18,2099-12-31,32809,Apixaban
2,3,5,4189269,2019-07-18,2099-12-31,32809,Bisoprolol
3,4,6,4186985,2019-07-24,2020-03-02,32809,Rivaroxaban
4,5,6,4187146,2019-09-01,2099-12-31,32809,Amiodaron


In [24]:
# device_specs table

# An AFBSTEP companion table: it records the device detail behind an episode
# that stock OMOP's device_exposure has no field for - the detection algorithm,
# its version, the serial number and the hardware version.
#
# Keyed by episode, not by device: the serial number repeats across every
# episode of the same device, but the algorithm version does not - it is what
# changes when the device takes a firmware update partway through.

specs = []
for _, m in mon[mon.dev_ref != ""].iterrows():
    d = ini_devices[ini_devices.dev_id == m.dev_ref]
    if d.empty or m.rec_id not in episode_of:
        continue                          # skip a record naming an unknown device, or one that produced no episode
    d = d.iloc[0]
    specs.append({
        "episode_id": episode_of[m.rec_id],              # the episode this device produced
        "person_id": person_id[m.pat_id],                # Foreign key to the person table
        "device_exposure_id": exposure_of[m.dev_ref],    # the device_exposure row describing the device itself
        "device_algorithm": d.algo_name,                 # AF detection algorithm; burden figures are not comparable across algorithms
        # the version reported for this record wins over the shipped one
        "device_algorithm_v": m.algo_version or d.algo_version,
        "device_sn": d.serial_no,                        # serial number, identifying the individual unit
        "device_v": d.hw_version,                        # hardware/firmware version of that unit
        "device_source_value": d.dev_kind,               # the site's own name for the device kind
    })

# one row per episode, so a window described by several source records collapses
device_specs = pd.DataFrame(specs).drop_duplicates(subset=["episode_id"])
device_specs.head()

,episode_id,person_id,device_exposure_id,device_algorithm,device_algorithm_v,device_sn,device_v,device_source_value
0,1,1,1,,,SN-773011,1.2,PM
2,2,1,1,,,SN-773011,1.2,PM
4,3,1,2,AF-Detect-Pro,2.3,SN-481920,1.0,PPG-WATCH
5,4,1,2,AF-Detect-Pro,2.3,SN-481920,1.0,PPG-WATCH
6,5,1,2,AF-Detect-Pro,2.4,SN-481920,1.0,PPG-WATCH


In [25]:
# source table (external file references)

# An AFBSTEP companion table. Raw files - device reports, scanned PDFs - are not
# held in the CDM. Instead a source row points at where the file is stored and
# says which CDM row it documents.
#
# That pointer is a pair: source_event_id holds the key, and
# source_event_field_concept_id names which table's field that key belongs to.
# The pair is needed because one file may document an episode, another a
# measurement, another a drug, and a bare id would not say which.
#
# Linking granularity therefore differs by modality: AF monitoring files
# document a whole episode, everything else documents one record.

episode_by_span = {(e.pat_id, e.ts_start, e.ts_end, e.dev_ref): e.episode_id
                   for _, e in epi.iterrows()}

sources = []

# device and monitoring files, attached to the episode they document
for _, m in fu[fu.rec_kind == "file"].iterrows():
    span = (m.pat_id, m.ts_start, m.ts_end, m.dev_ref)
    sources.append({"person_id": person_id[m.pat_id], "link_to_file": m.file_path,
                    "file_concept_id": FILE_KIND.get(m.file_kind, UNMAPPED),
                    "source_event_id": episode_by_span.get(span),
                    "source_event_field_concept_id": EPISODE_FIELD})

for _, r in ini_baseline.iterrows():
    pid = person_id[r.pat_id]

    # a scanned lab report documents whichever lab value it accompanied, found
    # through the person|source_value lookup built in the measurement cell
    if r.lab_doc:
        key = f"{pid}|hb_gl" if r.hb_gl else f"{pid}|krea_mgdl"
        sources.append({"person_id": pid, "link_to_file": r.lab_doc,
                        "file_concept_id": FILE_KIND["scan-pdf"],
                        "source_event_id": measurement_of_source.get(key),
                        "source_event_field_concept_id": MEASUREMENT_FIELD})

    # a scanned EHRA form documents the EHRA observation instead, so it is
    # looked up in the observation table built above
    if r.ehra_doc:
        match = observation[(observation.person_id == pid)
                            & (observation.observation_concept_id == EHRA_QUESTION)]
        sources.append({"person_id": pid, "link_to_file": r.ehra_doc,
                        "file_concept_id": FILE_KIND["scan-pdf"],
                        "source_event_id": match.observation_id.iloc[0]
                                           if len(match) else None,
                        "source_event_field_concept_id": OBSERVATION_FIELD})

# a scanned prescription documents the drug_exposure row it came from
for _, d in drugs[drugs.doc != ""].iterrows():
    sources.append({"person_id": person_id[d.pat_id], "link_to_file": d.doc,
                    "file_concept_id": FILE_KIND["scan-pdf"],
                    "source_event_id": d.drug_exposure_id,
                    "source_event_field_concept_id": DRUG_FIELD})

# assemble the table and mint the primary key
source_table = pd.DataFrame(sources)
source_table.insert(0, "source_id", range(1, len(source_table) + 1))
print(f"{len(source_table)} external file references")
source_table.head()

8 external file references


,source_id,person_id,link_to_file,file_concept_id,source_event_id,source_event_field_concept_id
0,1,1,./ext/CIED_P001_W1_report.pdf,2000010002,1,798885
1,2,1,./ext/CIED_P001_W1_transmission.json,2000010004,1,798885
2,3,16,./ext/CIED_P016_interrogation.pdf,2000010005,26,798885
3,4,17,./ext/HOLTER_P017_raw.edf,2000010003,27,798885
4,5,24,./ext/WEAR_P024_symptom_memo.m4a,2000010001,35,798885


In [26]:
# death table

# One row per person who died. OMOP keys this table by person rather than by a
# surrogate id, since a person dies once.
#
# The source records a cause as a free-text category, which is mapped to a
# concept through DEATH_CAUSE (defined in step 3). A cause that does not map
# becomes UNMAPPED rather than being dropped: the death is still a fact, and
# cause_source_value keeps what the site actually wrote.

dead = fu[fu.rec_kind == "death"]
death = pd.DataFrame({
    "person_id": dead.pat_id.map(person_id),                                   # Foreign key to the person table
    "death_date": dead.ts_start,                                               # Date of death
    "death_type_concept_id": DEATH_TYPE,                                       # Provenance of the record itself (defined in step 3)
    "cause_concept_id": dead.value.map(DEATH_CAUSE).fillna(UNMAPPED).astype(int),  # Cause of death mapped to a concept, or UNMAPPED if the category is not one we map
    "cause_source_value": dead.value,                                          # The cause as the site recorded it, kept so the mapping can be audited
})
death

,person_id,death_date,death_type_concept_id,cause_concept_id,cause_source_value
10,1,2024-11-14,44803317,2000006001,"cardiovascular, unspec."
19,3,2024-08-22,44803317,443392,malignancy
95,9,2019-08-03,44803317,4329847,MI
194,22,2022-11-30,44803317,2000006002,stroke


In [27]:
# study, study_attribute and person_study tables

# Three AFBSTEP companion tables, all built from one source table. The study
# information arrives long and mixed: entry_kind says what each row is, so the
# same file carries study metadata, multi-select attributes, arm definitions and
# per-person enrolments together. Each entry_kind is pulled out in turn below.
#
# The three tables answer different questions:
# - study            : one row per contributing study, its metadata
# - study_attribute  : attributes that allow more than one answer per study,
#                      which is why they cannot be columns on study itself
# - person_study     : which person was in which study, and in which arm

# study metadata is long (one row per key), so pivot it back to one row per study
meta = (ini_study[ini_study.entry_kind == "meta"]
        .pivot(index="study_id", columns="entry_key", values="entry_value")
        .reset_index())

# map the source's study identifier onto a small integer key
study_id = {s: i for i, s in enumerate(sorted(meta.study_id), start=1)}

study = pd.DataFrame({
    "study_id": meta.study_id.map(study_id),                        # Unique identifier for each study
    "study_name": meta.study_name,                                  # The study's name
    "nct_number": meta.nct,                                         # ClinicalTrials.gov registration, where the study has one
    "contributor": meta.contributor,                                # Who contributed the data
    "extended_affiliate": meta.ext_affiliate,                       # Whether the contributor is an extended affiliate of the consortium
    "extended_affiliate_detail": meta.ext_affiliate_name,           # Free text naming that affiliate
    "study_type_concept_id": meta.study_kind.map(STUDY_KIND),       # RCT, observational cohort, registry and so on (defined in step 3)
    "n_data_subjects": meta.n_subjects,                             # How many subjects the study contributes
    "study_start_date": meta.data_from,                             # First date the contributed data covers
    "study_end_date": meta.data_to,                                 # Last date the contributed data covers
    "median_follow_up_months": meta.median_fu_months,               # Median follow-up, in months
    "dataset_status_concept_id": meta.ds_status.map(DS_STATUS),     # Locked or rolling (defined in step 3)
    "transfer_frequency_concept_id": meta.transfer_mode.map(TRANSFER),  # One-off or periodic transfer (defined in step 3)
})

# Country of data collection and data provision level can each hold several
# answers for one study, so they get a row apiece here rather than a column on
# study. attribute_type_concept_id says which of the two an answer belongs to.
attrs = ini_study[ini_study.entry_kind.isin(["country", "provision"])]
study_attribute = pd.DataFrame({
    "study_id": attrs.study_id.map(study_id),                       # Foreign key to the study table
    "attribute_type_concept_id": attrs.entry_kind.map(ATTR_TYPE),   # Which attribute this row answers: country, or provision level
    "value_concept_id": [                                           # The answer itself, drawn from whichever vocabulary that attribute uses
        (COUNTRY if k == "country" else PROVISION).get(v, UNMAPPED)
        for k, v in zip(attrs.entry_kind, attrs.entry_value)],
})

# Each study names its own arms in its own words, and a separate row records
# what role each name plays. This lookup turns a study's free-text arm name into
# that controlled role, so "Arm B" at one site and "control" at another compare.
arm_role = {(r.study_id, r.entry_value): r.entry_note
            for r in ini_study[ini_study.entry_kind == "arm"].itertuples()}

enrol = ini_study[ini_study.entry_kind == "enrollment"]
person_study = pd.DataFrame({
    "person_id": enrol.pat_id.map(person_id),                       # Foreign key to the person table
    "study_id": enrol.study_id.map(study_id),                       # Foreign key to the study table
    "observation_period_id": enrol.pat_id.map(period_of),           # The observation period this enrolment covers
    "enrollment_date": enrol.entry_note,                            # When the person entered the study
    "arm_source_value": enrol.entry_value,                          # The arm name as the study wrote it
    "arm_type_concept_id": [                                        # That arm's controlled role: treatment, control, placebo comparator and so on
        ARM_ROLE.get(arm_role.get((s, a), ""), UNMAPPED)
        for s, a in zip(enrol.study_id, enrol.entry_value)],
})
print(f"{len(study)} studies, {len(study_attribute)} attributes, "
      f"{len(person_study)} enrolments")
person_study.head()

6 studies, 15 attributes, 23 enrolments


,person_id,study_id,observation_period_id,enrollment_date,arm_source_value,arm_type_concept_id
19,1,1,1,2024-02-01,Physical Intervention,37529738
20,2,1,2,2024-02-03,Clinical Intervention,37529738
21,3,1,3,2024-02-05,Control,37542561
38,5,2,4,2019-07-18,Arm A - Ablation,37529738
39,6,2,5,2019-07-24,Arm B - Placebo,37530454


# Step 6: write the CDM tables

In [28]:
# Write every built table to CSV, one file per CDM table.
#
# The validator reads a directory of CSVs, so this is the point where the
# in-memory frames become the export a partner would actually submit.

def write_table(frame, name):
    """Write one CDM table, with every declared field in DDL order.

    Any id column holding a missing value becomes float64 in pandas, which
    writes `1.0` instead of `1` and silently breaks every foreign key that
    points at it. Nullable Int64 keeps ids as integers.
    """
    # reindex against the schema so declared-but-unused fields are written empty
    # rather than omitted, giving the file the shape of a real export
    out = frame.reindex(columns=list(schema.tables[name].columns))
    for column in out.columns:
        if column.endswith("_id") or column.endswith("_concept_id"):
            out[column] = pd.to_numeric(out[column], errors="coerce").astype("Int64")
    out.to_csv(OUT / f"{name}.csv", index=False)


# every table built above, keyed by its CDM name
tables = {"person": person, "observation_period": observation_period,
          "visit_occurrence": visit_occurrence,
          "condition_occurrence": condition_occurrence,
          "procedure_occurrence": procedure_occurrence,
          "device_exposure": device_exposure, "drug_exposure": drug_exposure,
          "measurement": measurement, "observation": observation,
          "episode": episode, "episode_event": episode_event,
          "device_specs": device_specs, "source": source_table,
          "death": death, "study": study, "study_attribute": study_attribute,
          "person_study": person_study}

OUT.mkdir(parents=True, exist_ok=True)
for name, frame in tables.items():
    write_table(frame, name)
    print(f"  {name:22s} {len(frame):>5} rows")

  person                    26 rows
  observation_period        23 rows
  visit_occurrence          25 rows
  condition_occurrence      72 rows
  procedure_occurrence      17 rows
  device_exposure           30 rows
  drug_exposure             29 rows
  measurement              313 rows
  observation              350 rows
  episode                   37 rows
  episode_event              8 rows
  device_specs              37 rows
  source                     8 rows
  death                      4 rows
  study                      6 rows
  study_attribute           15 rows
  person_study              23 rows


# Step 7: check the mapping against the codebook, then validate

The first four persons reproduce the worked example in
`mapping_codebook.md` exactly, so the mapping can be checked against a
known answer before the validator is asked for an opinion.

In [29]:
# Check the mapping against the published codebook.
#
# `mapping_guideline/mapping_codebook.md` works through patients P-001..P-003 by
# hand and states what each one should look like once mapped. This cell asserts
# that the pipeline above actually produced that. It is what keeps the worked
# example and the written guidance from drifting apart: change a mapping rule in
# one and this cell fails until the other is updated too.
#
# Validation (the next cell) asks whether the export is well formed. This asks
# something different and stricter: whether it says the right thing.

# P-001..P-003 have a published expected mapping. Assert it.
p1, p2, p3 = person_id["P-001"], person_id["P-002"], person_id["P-003"]


def one(frame, **where):
    """Return the rows of `frame` matching every column=value pair given."""
    m = frame
    for k, v in where.items():
        m = m[m[k] == v]
    return m


# D-0002 is the wearable; its rows are pulled out once and reused below
wearable_specs = device_specs[
    device_specs.device_exposure_id == exposure_of["D-0002"]]

checks = {
    # --- demographics ---
    "person P-001 is FEMALE":
        one(person, person_id=p1).gender_concept_id.iloc[0] == 8532,
    "person P-003 race is Asian Indian":
        one(person, person_id=p3).race_concept_id.iloc[0] == 38003574,

    # --- the three ways a historical comorbidity can be dated ---
    "P-001 heart failure confirmed at baseline (dating Case 1)":
        len(one(condition_occurrence, person_id=p1, condition_concept_id=316139,
                condition_start_date="2024-02-01")) == 1,
    "P-002 heart failure dated to the true event (dating Case 2)":
        len(one(condition_occurrence, person_id=p2, condition_concept_id=316139,
                condition_start_date="2019-11-03")) == 1,
    "P-003 heart failure uses the 1900-01-01 sentinel (dating Case 3)":
        len(one(condition_occurrence, person_id=p3, condition_concept_id=316139,
                condition_start_date="1900-01-01")) == 1,
    "P-001 peri-procedural complication":
        len(one(condition_occurrence, person_id=p1,
                condition_concept_id=2000005000)) == 1,

    # --- the five missingness cases, each landing somewhere different ---
    "P-002 diabetes is explicitly negative (missingness Case 2)":
        len(one(observation, person_id=p2, observation_concept_id=201820,
                value_as_concept_id=4189457)) == 1,
    "P-002 LVEF not performed (missingness Case 3)":
        len(one(measurement, person_id=p2, measurement_concept_id=3027172,
                value_as_concept_id=45884199)) == 1,
    "P-001 mEHRA unable to determine (missingness Case 4)":
        len(one(observation, person_id=p1, observation_concept_id=2000009007,
                value_as_concept_id=45880382)) == 1,
    "P-002 mEHRA missing cause unknown (missingness Case 5)":
        len(one(observation, person_id=p2, observation_concept_id=2000009007,
                value_as_concept_id=2000013000)) == 1,
    # Case 1 is the one with no marker: never asked means no row at all, so it
    # can only be checked by an absence
    "P-001 has no LVEF row at all (missingness Case 1)":
        len(one(measurement, person_id=p1, measurement_concept_id=3027172)) == 0,

    # --- AF burden: the two episode kinds and the values hanging off them ---
    "P-001 two aggregated windows":
        len(one(episode, person_id=p1,
                episode_concept_id=2000011002)) == 2,
    "P-001 four disaggregated episodes":
        len(one(episode, person_id=p1,
                episode_concept_id=2000011003)) == 4,
    "P-001 AF burden 4.04 % and 12.8 %":
        sorted(one(measurement, person_id=p1,
                   measurement_concept_id=2000008002).value_as_number.tolist())
        == ["12.8", "4.04"],
    "P-001 peak rates 188/165/201/176":
        sorted(one(measurement, person_id=p1,
                   measurement_concept_id=2000009004).value_as_number.tolist())
        == ["165", "176", "188", "201"],
    "P-001 episode 2 is atrial flutter":
        len(one(observation, person_id=p1, observation_concept_id=2000009003,
                value_as_concept_id=314665)) == 1,

    # --- devices ---
    # the codebook's device_specs example is the wearable's four episodes;
    # the two CIED windows carry their own device_specs rows.
    "P-001 wearable firmware changes 2.3 -> 2.4 across its episodes":
        sorted(wearable_specs.device_algorithm_v.unique()) == ["2.3", "2.4"],
    "P-001 wearable serial number is constant across all four":
        set(wearable_specs.device_sn) == {"SN-481920"} and len(wearable_specs) == 4,
    "P-003 Holter has a real end date":
        one(device_exposure, person_id=p3,
            device_concept_id=45762052).device_exposure_end_date.iloc[0]
        == "2024-03-22",
    "P-001 ablation energy source is pulsed-field":
        len(one(device_exposure, person_id=p1,
                device_concept_id=2000004007)) == 1,

    # --- death, including the person who did not die ---
    "P-001 death is cardiovascular":
        one(death, person_id=p1).cause_concept_id.iloc[0] == 2000006001,
    "P-003 death is a malignant neoplasm":
        one(death, person_id=p3).cause_concept_id.iloc[0] == 443392,
    "P-002 has no death row":
        len(one(death, person_id=p2)) == 0,

    # --- edge cases the codebook calls out deliberately ---
    # P-004 was screened but never enrolled, so nothing downstream should exist
    "P-004 has no observation period":
        person_id["P-004"] not in set(observation_period.person_id),
    "P-004 has no person_study row":
        person_id["P-004"] not in set(person_study.person_id),
    "P-001 window 1 is documented by two files":
        len(source_table[(source_table.source_event_field_concept_id == 798885)
                         & (source_table.source_event_id
                            == episode_of["M-0001"])]) == 2,
    "every episode-linked source row resolves to a real episode":
        set(source_table[source_table.source_event_field_concept_id == 798885]
            .source_event_id.dropna()) <= set(episode.episode_id),

    # P-001's episodes came from two different devices, so none of them nests
    # inside another and no parent-child link should have been written
    "no episode_event links for P-001 (different devices)":
        not set(episode_event.episode_id) & set(one(episode, person_id=p1).episode_id),
}

failed = [k for k, ok in checks.items() if not ok]
for k, ok in checks.items():
    print(f"  {'ok  ' if ok else 'FAIL'}  {k}")
print(f"\n{len(checks) - len(failed)}/{len(checks)} codebook checks pass")
assert not failed, failed

  ok    person P-001 is FEMALE
  ok    person P-003 race is Asian Indian
  ok    P-001 heart failure confirmed at baseline (dating Case 1)
  ok    P-002 heart failure dated to the true event (dating Case 2)
  ok    P-003 heart failure uses the 1900-01-01 sentinel (dating Case 3)
  ok    P-001 peri-procedural complication
  ok    P-002 diabetes is explicitly negative (missingness Case 2)
  ok    P-002 LVEF not performed (missingness Case 3)
  ok    P-001 mEHRA unable to determine (missingness Case 4)
  ok    P-002 mEHRA missing cause unknown (missingness Case 5)
  ok    P-001 has no LVEF row at all (missingness Case 1)
  ok    P-001 two aggregated windows
  ok    P-001 four disaggregated episodes
  ok    P-001 AF burden 4.04 % and 12.8 %
  ok    P-001 peak rates 188/165/201/176
  ok    P-001 episode 2 is atrial flutter
  ok    P-001 wearable firmware changes 2.3 -> 2.4 across its episodes
  ok    P-001 wearable serial number is constant across all four
  ok    P-003 Holter has a real en

In [30]:
# Validate the export against the AFBSTEP-OMOP specification.
#
# This reads the CSVs written above and checks them in five layers: conformance,
# referential integrity, terminology, plausibility and the minimal data set.
# An `error` means a stated requirement is unmet and the dataset does not pass;
# a `warning` marks something the specification permits but a human should see.
#
# Concepts here are resolved against the registry pinned in the package. To
# additionally confirm that every concept id really exists upstream, and is
# neither retired nor in the wrong domain, build a local vocabulary index and
# run `--validate-full` instead - see the README.
report = validate(read_dataset(OUT, spec), schema, spec, vocabulary)
print(report.summary())

AFBSTEP validation: PASSED — 0 error(s), 30 warning(s), 18 info [concepts checked against: pinned specification (272 concepts)]

Tables read:
  condition_occurrence: 72 rows
  death: 4 rows
  device_exposure: 30 rows
  device_specs: 37 rows
  drug_exposure: 29 rows
  episode: 37 rows
  episode_event: 8 rows
  measurement: 313 rows
  observation: 350 rows
  observation_period: 23 rows
  person: 26 rows
  person_study: 23 rows
  procedure_occurrence: 17 rows
  source: 8 rows
  study: 6 rows
  study_attribute: 15 rows
  visit_occurrence: 25 rows

Warnings — permitted, but worth checking:
  [warning] terminology: condition_occurrence.condition_concept_id: concept id 0: source value did not map (16 rows)
  [warning] terminology: condition_occurrence.condition_status_concept_id: 1 concept(s) not in pinned specification (272 concepts), e.g. [32893]; existence and domain were not verified (31 rows)
  [warning] terminology: device_exposure.device_concept_id: 1 concept(s) not in pinned specifica